# 00 — Setup Iceberg Catalog

Run this notebook **once** after `make up` and `make nessie-init-bucket`.

It verifies:
1. The `iceberg-warehouse` S3 bucket is accessible in RustFS
2. The Nessie REST catalog is reachable
3. The `demo` namespace exists (creates it if not)

All connection parameters come from environment variables injected by Docker Compose.

In [2]:
import os

import boto3
from botocore.config import Config
from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

print(f"Nessie URI    : {NESSIE_URI}")
print(f"S3 endpoint   : {S3_ENDPOINT}")
print(f"Warehouse URI : {WAREHOUSE_URI}")

Nessie URI    : http://nessie:19120/iceberg
S3 endpoint   : http://rustfs:9000
Warehouse URI : s3://iceberg-warehouse/warehouse


## 1. Verify S3 bucket

In [3]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    config=Config(s3={"addressing_style": "path"}),
    region_name="us-east-1",
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print("Buckets:", buckets)

assert WAREHOUSE_BUCKET in buckets, (
    f"Bucket '{WAREHOUSE_BUCKET}' not found. Run: make nessie-init-bucket"
)
print("✔ S3 bucket check passed.")

Buckets: ['iceberg-warehouse']
✔ S3 bucket check passed.


## 2. Connect to Nessie REST catalog

In [4]:
catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("✔ Connected to catalog:", catalog.name)

✔ Connected to catalog: nessie


## 3. Create `demo` namespace

In [5]:
try:
    catalog.create_namespace("demo")
    print("Namespace 'demo' created.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Namespace 'demo' already exists — skipping.")
    else:
        raise

print("Namespaces:", catalog.list_namespaces())

Namespace 'demo' created.
Namespaces: [('demo',)]
